# Estimate Mode Choice 2

Estimate mode choice models of TNC vs transit/walk

Uses combined HH travel survey + TNC data.  Updates availability from version 1, focusing on weighted estimation. 

In [22]:
import numpy as np

import pandas as pd

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.reset_option('display.float_format')

import biogeme.biogeme as bio
import biogeme.database as biodb
from biogeme import models
from biogeme.expressions import Beta, Variable

In [23]:
# read the data
df = pd.read_csv('out/combined_estimation_file.csv')
df.head()

C:\Users\ger225\AppData\Local\Temp\ipykernel_30840\360866534.py:2: DtypeWarning: Columns (0: depart_date, 1: linked_trip_mode_labeled, 2: income_labeled) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('out/combined_estimation_file.csv')


,Unnamed: 0,hh_id,person_id,person_num,day_id,day_num,depart_date,o_tract_2020,d_tract_2020,linked_trip_id,linked_trip_num,linked_trip_mode,linked_trip_weight,linked_trip_mode_labeled,mode,mode2,distance_miles,duration_minutes,o_district,d_district,o_community,d_community,time_period,ff_car_time_minutes,car_ivt,tnc_wait,tnc_time,tnc_fare,transit_fare,walk_time,transit_time,transit_or_walk_time,walk_faster_than_transit,transit_or_walk_fare,income_broad,income_labeled,hh_share_inc_under_100k,hh_share_inc_over_100k,hh_share_inc_under_100k_dropoff,hh_share_inc_over_100k_dropoff,tnc_trip_id,obs_fare,obs_tip,obs_additional_charges,transit_avail,walk_avail,transit_or_walk_avail,tnc_time_2,tnc_fare_2,time_period_num,avg_obs_tnc_time,avg_obs_tnc_fare,tnc_observed,tnc_time_3,tnc_fare_3,tnc_time_minus_transit_walk,tnc_cost_minus_transit_walk
0,0,24000124.0,2.400012e+09,1.0,2.400012e+11,1.0,2024-05-21,17031320101,17031081500,2.400012e+15,1.0,15.0,1853.792592,Walk,walk,walk,0.810270,20.0,Downtown,Downtown,32.0,8.0,midday,3.738333,6.186942,5,11.186942,6.707731,2.5,16.205401,22.0,16.205401,True,0.0,5.0,"$150,000 or more",0.341000,0.659000,0.321678,0.678322,NaN,NaN,NaN,NaN,1,1,1,11.186942,6.707731,3,NaN,NaN,False,11.186942,6.707731,-5.018459,6.707731
1,1,24000124.0,2.400012e+09,1.0,2.400012e+11,1.0,2024-05-21,17031081500,17031081403,2.400012e+15,2.0,15.0,1853.792592,Walk,walk,walk,0.338027,28.0,Downtown,Downtown,8.0,8.0,midday,1.660000,2.747300,5,7.747300,4.798943,2.5,6.760535,7.0,6.760535,True,0.0,5.0,"$150,000 or more",0.321678,0.678322,0.460539,0.539461,NaN,NaN,NaN,NaN,1,1,1,7.747300,4.798943,3,7.157143,9.642857,True,12.157143,9.642857,5.396607,9.642857
2,2,24000124.0,2.400012e+09,1.0,2.400012e+11,1.0,2024-05-21,17031081403,17031320101,2.400012e+15,3.0,15.0,1853.792592,Walk,walk,walk,0.549293,15.0,Downtown,Downtown,8.0,32.0,midday,3.421667,5.662858,5,10.662858,6.244886,2.5,10.985870,23.0,10.985870,True,0.0,5.0,"$150,000 or more",0.460539,0.539461,0.341000,0.659000,NaN,NaN,NaN,NaN,1,1,1,10.662858,6.244886,3,NaN,NaN,False,10.662858,6.244886,-0.323012,6.244886
3,3,24000124.0,2.400012e+09,1.0,2.400012e+11,1.0,2024-05-21,17031320101,17031320101,2.400012e+15,4.0,15.0,1853.792592,Walk,walk,walk,0.319386,16.0,Downtown,Downtown,32.0,32.0,midday,2.201667,3.643758,5,8.643758,5.167457,2.5,6.387712,12.0,6.387712,True,0.0,5.0,"$150,000 or more",0.341000,0.659000,0.341000,0.659000,NaN,NaN,NaN,NaN,1,1,1,8.643758,5.167457,3,NaN,NaN,False,8.643758,5.167457,2.256047,5.167457
4,4,24000124.0,2.400012e+09,2.0,2.400012e+11,1.0,2024-05-21,17031320101,17031320102,2.400012e+15,1.0,15.0,1853.792592,Walk,walk,walk,0.751861,16.0,Downtown,Downtown,32.0,32.0,midday,2.201667,3.643758,5,8.643758,5.561010,2.5,15.037220,12.0,12.000000,False,2.5,5.0,"$150,000 or more",0.341000,0.659000,0.519520,0.480480,NaN,NaN,NaN,NaN,1,1,1,8.643758,5.561010,3,NaN,NaN,False,8.643758,5.561010,-3.356242,3.061010


In [24]:
# which columns have NaNs, and how many
df.isna().sum()


Unnamed: 0                              0
hh_id                              156150
person_id                          156150
person_num                         156150
day_id                             156150
day_num                            156150
depart_date                        156150
o_tract_2020                            0
d_tract_2020                            0
linked_trip_id                     156150
linked_trip_num                    156150
linked_trip_mode                   156150
linked_trip_weight                      0
linked_trip_mode_labeled           156150
mode                                    0
mode2                                   0
distance_miles                          0
duration_minutes                        0
o_district                              0
d_district                              0
o_community                             0
d_community                             0
time_period                             0
ff_car_time_minutes               

In [25]:
# drop recrods where we don't know the income shares

trips_before = len(df)

df = df[df['hh_share_inc_under_100k']>0]
df = df[df['hh_share_inc_under_100k']>0]

print("Before: " + str(trips_before) + " After: " + str(len(df)))

Before: 161249 After: 161105


In [26]:
# fill the remaining missing values with zeros to make biogeme happy--BE CAREFUL!
df = df.fillna(0)

In [27]:
# add a flag for trips made by people in HHs with <$100k, $100k+ and missing annual income
# income_broad: 
# 1	Under $30,000
# 2	$30,000-$59,999
# 3	$60,000-$99,999
# 4	$100,000-$149,999
# 5	$150,000 or more
# 999	Prefer not to answer

df['inc_under_100k'] = np.where(df['income_broad']<=3, 1, 0)
df['inc_over_100k']  = np.where((df['income_broad']==4) | (df['income_broad']==5), 1, 0)
df['inc_missing']    = np.where((df['income_broad']==999), 1, 0)

In [28]:
# update availability

# transit must be less than 2 hours
df['transit_avail'] = np.where(df['transit_time']<120, 1, 0)

# walk must be less than 30 minutes
df['walk_avail'] = np.where(df['walk_time']<30, 1, 0)

# drop trips that choose an unavailable alternative
trips_before = len(df)
df = df[(df['transit_avail']==1) | (df['mode']!='transit')].copy()
df = df[(df['walk_avail']==1) | (df['mode']!='walk')].copy()

print("Before: " + str(trips_before) + " After: " + str(len(df)))

Before: 161105 After: 160922


In [29]:
# define a flag for the origin and destination downtown

df['o_downtown'] = np.where(df['o_district']=='Downtown', 1, 0)
df['d_downtown'] = np.where(df['d_district']=='Downtown', 1, 0)


In [30]:
# calculate normalized weights
df['normalized_weights'] = df['linked_trip_weight'] / df['linked_trip_weight'].sum() * len(df)

In [31]:
# Biogeme needs a NUMERIC choice column: tnc=1, transit=2, walk=3 and only numeric values in its database format. 
df['CHOICE'] = df['mode'].map({'tnc': 1, 'transit': 2, 'walk': 3})
df['BINARY_CHOICE'] = df['mode'].map({'tnc': 1, 'transit': 2, 'walk': 2})

df_numeric = df.select_dtypes(include='number').copy()

db = biodb.Database('mode_choice', df_numeric)

# Starting Model 

This is from Estimate Combined Mode Choice 2.ipynb.  It is what we previously marked the winner.  It is good, but when I look at the application, I see that I'm under-estimating ride-hailing trips within the downtown area.  Try a downtown ASC to fix that. 

In [32]:
# add our best estimate of TNC time and fare

# include the transit fare
# weighted estimation with zonal incomes
# Add constant segmented by income
# use observed TNC time/fare where available

# --- coefficients ---
# Name, starting value, lower bound, upper bound, status (0=estimate, 1=fixed)
ASC_TRANSIT = Beta('ASC_TRANSIT', 0, None, None, 0)
ASC_WALK    = Beta('ASC_WALK',    0, None, None, 0)    
B_TIME      = Beta('B_TIME',      0, None, None, 0)   # generic, shared across modes
B_COST_LOW  = Beta('B_COST_LOW',  0, None, None, 0)   # cost coefficient for lower income travelers
B_COST_HI   = Beta('B_COST_HI',   0, None, None, 0)   # cost coefficient for higher income travelers

# --- variables ---
tnc_time     = Variable('tnc_time_3')
transit_time = Variable('transit_time')
walk_time    = Variable('walk_time')
tnc_fare     = Variable('tnc_fare_3')
transit_fare = Variable('transit_fare')

hh_share_inc_under_100k = Variable('hh_share_inc_under_100k')
hh_share_inc_over_100k = Variable('hh_share_inc_over_100k')

CHOICE       = Variable('CHOICE')

# --- utility equations ---
V_tnc     =               B_TIME * tnc_time     + B_COST_LOW * tnc_fare * hh_share_inc_under_100k + B_COST_HI * tnc_fare * hh_share_inc_over_100k 
V_transit = ASC_TRANSIT + B_TIME * transit_time + B_COST_LOW * transit_fare * hh_share_inc_under_100k + B_COST_HI * transit_fare * hh_share_inc_over_100k 
V_walk    = ASC_WALK    + B_TIME * walk_time    

# specify which equations align with which alternatives, and the availability of each alternative
V  = {1: V_tnc, 2: V_transit, 3: V_walk}
avail = {1: 1, 2: Variable('transit_avail'), 3: Variable('walk_avail')}         

# --- estimate ---
logprob = models.loglogit(V, avail, CHOICE)
the_biogeme = bio.BIOGEME(db, {'loglike' : logprob, 'weight' : Variable('normalized_weights')})
the_biogeme.modelName = 'mnl_mode_choice'
the_biogeme.calculate_null_loglikelihood(avail=avail)
results = the_biogeme.estimate()

# --- print results ---
print(results.short_summary())
print(results.get_estimated_parameters())

# --- print value of time ---
params = results.get_estimated_parameters()
vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_LOW']['Value']
print("\nValue of Time for HH <$100k: " + str(round(vot, 2)))

vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_HI']['Value']
print("Value of Time for HH $100k+: " + str(round(vot, 2)))

Results for model mnl_mode_choice
Nbr of parameters:		5
Sample size:			160922
Excluded data:			0
Null log likelihood:		-118890.3
Final log likelihood:		-64088.56
Likelihood ratio test (null):		109603.4
Rho square (null):			0.461
Rho bar square (null):			0.461
Akaike Information Criterion:	128187.1
Bayesian Information Criterion:	128237.1

                Value  Rob. Std err  Rob. t-test  Rob. p-value
ASC_TRANSIT  1.706168      0.025159    67.815065           0.0
ASC_WALK     3.520829      0.028359   124.151073           0.0
B_COST_HI   -0.039206      0.003356   -11.682228           0.0
B_COST_LOW  -0.077116      0.003006   -25.652129           0.0
B_TIME      -0.021315      0.000657   -32.428572           0.0

Value of Time for HH <$100k: 16.58
Value of Time for HH $100k+: 32.62


# With downtown constant.

Same model, but test some variations of a downtown constant. 

In [33]:
# include a constant for trips within downtown

# include the transit fare
# weighted estimation with zonal incomes
# Add constant segmented by income
# use observed TNC time/fare where available

# --- coefficients ---
# Name, starting value, lower bound, upper bound, status (0=estimate, 1=fixed)
ASC_TRANSIT = Beta('ASC_TRANSIT', 0, None, None, 0)
ASC_WALK    = Beta('ASC_WALK',    0, None, None, 0)    
B_TIME      = Beta('B_TIME',      0, None, None, 0)   # generic, shared across modes
B_COST_LOW  = Beta('B_COST_LOW',  0, None, None, 0)   # cost coefficient for lower income travelers
B_COST_HI   = Beta('B_COST_HI',   0, None, None, 0)   # cost coefficient for higher income travelers

ASC_TNC_DOWNTOWN = Beta('ASC_TNC_DOWNTOWN', 0, None, None, 0) 

# --- variables ---
tnc_time     = Variable('tnc_time_3')
transit_time = Variable('transit_time')
walk_time    = Variable('walk_time')
tnc_fare     = Variable('tnc_fare_3')
transit_fare = Variable('transit_fare')

hh_share_inc_under_100k = Variable('hh_share_inc_under_100k')
hh_share_inc_over_100k = Variable('hh_share_inc_over_100k')

o_downtown = Variable('o_downtown')
d_downtown = Variable('d_downtown')

CHOICE       = Variable('CHOICE')

# --- utility equations ---
V_tnc     =               B_TIME * tnc_time     + B_COST_LOW * tnc_fare * hh_share_inc_under_100k + B_COST_HI * tnc_fare * hh_share_inc_over_100k          + ASC_TNC_DOWNTOWN * o_downtown * d_downtown
V_transit = ASC_TRANSIT + B_TIME * transit_time + B_COST_LOW * transit_fare * hh_share_inc_under_100k + B_COST_HI * transit_fare * hh_share_inc_over_100k 
V_walk    = ASC_WALK    + B_TIME * walk_time    

# specify which equations align with which alternatives, and the availability of each alternative
V  = {1: V_tnc, 2: V_transit, 3: V_walk}
avail = {1: 1, 2: Variable('transit_avail'), 3: Variable('walk_avail')}         

# --- estimate ---
logprob = models.loglogit(V, avail, CHOICE)
the_biogeme = bio.BIOGEME(db, {'loglike' : logprob, 'weight' : Variable('normalized_weights')})
the_biogeme.modelName = 'mnl_mode_choice'
the_biogeme.calculate_null_loglikelihood(avail=avail)
results = the_biogeme.estimate()

# --- print results ---
print(results.short_summary())
print(results.get_estimated_parameters())

# --- print value of time ---
params = results.get_estimated_parameters()
vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_LOW']['Value']
print("\nValue of Time for HH <$100k: " + str(round(vot, 2)))

vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_HI']['Value']
print("Value of Time for HH $100k+: " + str(round(vot, 2)))

Results for model mnl_mode_choice
Nbr of parameters:		6
Sample size:			160922
Excluded data:			0
Null log likelihood:		-118890.3
Final log likelihood:		-63651.1
Likelihood ratio test (null):		110478.4
Rho square (null):			0.465
Rho bar square (null):			0.465
Akaike Information Criterion:	127314.2
Bayesian Information Criterion:	127374.1

                     Value  Rob. Std err  Rob. t-test  Rob. p-value
ASC_TNC_DOWNTOWN  1.010445      0.032546    31.046484           0.0
ASC_TRANSIT       1.982917      0.028421    69.770458           0.0
ASC_WALK          3.801979      0.031513   120.647096           0.0
B_COST_HI        -0.046844      0.003634   -12.890054           0.0
B_COST_LOW       -0.059125      0.002929   -20.188760           0.0
B_TIME           -0.024780      0.000678   -36.562162           0.0

Value of Time for HH <$100k: 25.15
Value of Time for HH $100k+: 31.74


It's clearly significant.  My low-income VOT becomes higher than I would expect.  I suspect the trips within downtown are mostly non-home-based trips, so it's not clear there is a strong link to the income distribution there. 

In [34]:
# include a constant for just trips starting in downtown

# include the transit fare
# weighted estimation with zonal incomes
# Add constant segmented by income
# use observed TNC time/fare where available

# --- coefficients ---
# Name, starting value, lower bound, upper bound, status (0=estimate, 1=fixed)
ASC_TRANSIT = Beta('ASC_TRANSIT', 0, None, None, 0)
ASC_WALK    = Beta('ASC_WALK',    0, None, None, 0)    
B_TIME      = Beta('B_TIME',      0, None, None, 0)   # generic, shared across modes
B_COST_LOW  = Beta('B_COST_LOW',  0, None, None, 0)   # cost coefficient for lower income travelers
B_COST_HI   = Beta('B_COST_HI',   0, None, None, 0)   # cost coefficient for higher income travelers

ASC_TNC_DOWNTOWN = Beta('ASC_TNC_DOWNTOWN', 0, None, None, 0) 

# --- variables ---
tnc_time     = Variable('tnc_time_3')
transit_time = Variable('transit_time')
walk_time    = Variable('walk_time')
tnc_fare     = Variable('tnc_fare_3')
transit_fare = Variable('transit_fare')

hh_share_inc_under_100k = Variable('hh_share_inc_under_100k')
hh_share_inc_over_100k = Variable('hh_share_inc_over_100k')

o_downtown = Variable('o_downtown')
d_downtown = Variable('d_downtown')

CHOICE       = Variable('CHOICE')

# --- utility equations ---
V_tnc     =               B_TIME * tnc_time     + B_COST_LOW * tnc_fare * hh_share_inc_under_100k + B_COST_HI * tnc_fare * hh_share_inc_over_100k          + ASC_TNC_DOWNTOWN * o_downtown 
V_transit = ASC_TRANSIT + B_TIME * transit_time + B_COST_LOW * transit_fare * hh_share_inc_under_100k + B_COST_HI * transit_fare * hh_share_inc_over_100k 
V_walk    = ASC_WALK    + B_TIME * walk_time    

# specify which equations align with which alternatives, and the availability of each alternative
V  = {1: V_tnc, 2: V_transit, 3: V_walk}
avail = {1: 1, 2: Variable('transit_avail'), 3: Variable('walk_avail')}         

# --- estimate ---
logprob = models.loglogit(V, avail, CHOICE)
the_biogeme = bio.BIOGEME(db, {'loglike' : logprob, 'weight' : Variable('normalized_weights')})
the_biogeme.modelName = 'mnl_mode_choice'
the_biogeme.calculate_null_loglikelihood(avail=avail)
results = the_biogeme.estimate()

# --- print results ---
print(results.short_summary())
print(results.get_estimated_parameters())

# --- print value of time ---
params = results.get_estimated_parameters()
vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_LOW']['Value']
print("\nValue of Time for HH <$100k: " + str(round(vot, 2)))

vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_HI']['Value']
print("Value of Time for HH $100k+: " + str(round(vot, 2)))

Results for model mnl_mode_choice
Nbr of parameters:		6
Sample size:			160922
Excluded data:			0
Null log likelihood:		-118890.3
Final log likelihood:		-63964.52
Likelihood ratio test (null):		109851.5
Rho square (null):			0.462
Rho bar square (null):			0.462
Akaike Information Criterion:	127941
Bayesian Information Criterion:	128001

                     Value  Rob. Std err  Rob. t-test  Rob. p-value
ASC_TNC_DOWNTOWN  0.467195      0.031169    14.989296           0.0
ASC_TRANSIT       1.832696      0.026200    69.949449           0.0
ASC_WALK          3.621026      0.028817   125.656834           0.0
B_COST_HI        -0.063857      0.004094   -15.596604           0.0
B_COST_LOW       -0.064149      0.003020   -21.238860           0.0
B_TIME           -0.023223      0.000664   -34.972849           0.0

Value of Time for HH <$100k: 21.72
Value of Time for HH $100k+: 21.82


In [35]:
# try separate walk and transit constants within downtown

# include the transit fare
# weighted estimation with zonal incomes
# Add constant segmented by income
# use observed TNC time/fare where available

# --- coefficients ---
# Name, starting value, lower bound, upper bound, status (0=estimate, 1=fixed)
ASC_TRANSIT = Beta('ASC_TRANSIT', 0, None, None, 0)
ASC_WALK    = Beta('ASC_WALK',    0, None, None, 0)    
B_TIME      = Beta('B_TIME',      0, None, None, 0)   # generic, shared across modes
B_COST_LOW  = Beta('B_COST_LOW',  0, None, None, 0)   # cost coefficient for lower income travelers
B_COST_HI   = Beta('B_COST_HI',   0, None, None, 0)   # cost coefficient for higher income travelers

ASC_TRANSIT_DOWNTOWN = Beta('ASC_TRANSIT_DOWNTOWN', 0, None, None, 0) 
ASC_WALK_DOWNTOWN = Beta('ASC_WALK_DOWNTOWN', 0, None, None, 0) 

# --- variables ---
tnc_time     = Variable('tnc_time_3')
transit_time = Variable('transit_time')
walk_time    = Variable('walk_time')
tnc_fare     = Variable('tnc_fare_3')
transit_fare = Variable('transit_fare')

hh_share_inc_under_100k = Variable('hh_share_inc_under_100k')
hh_share_inc_over_100k = Variable('hh_share_inc_over_100k')

o_downtown = Variable('o_downtown')
d_downtown = Variable('d_downtown')

CHOICE       = Variable('CHOICE')

# --- utility equations ---
V_tnc     =               B_TIME * tnc_time     + B_COST_LOW * tnc_fare * hh_share_inc_under_100k + B_COST_HI * tnc_fare * hh_share_inc_over_100k         
V_transit = ASC_TRANSIT + B_TIME * transit_time + B_COST_LOW * transit_fare * hh_share_inc_under_100k + B_COST_HI * transit_fare * hh_share_inc_over_100k + ASC_TRANSIT_DOWNTOWN * o_downtown * d_downtown
V_walk    = ASC_WALK    + B_TIME * walk_time + ASC_WALK_DOWNTOWN * o_downtown * d_downtown   

# specify which equations align with which alternatives, and the availability of each alternative
V  = {1: V_tnc, 2: V_transit, 3: V_walk}
avail = {1: 1, 2: Variable('transit_avail'), 3: Variable('walk_avail')}         

# --- estimate ---
logprob = models.loglogit(V, avail, CHOICE)
the_biogeme = bio.BIOGEME(db, {'loglike' : logprob, 'weight' : Variable('normalized_weights')})
the_biogeme.modelName = 'mnl_mode_choice'
the_biogeme.calculate_null_loglikelihood(avail=avail)
results = the_biogeme.estimate()

# --- print results ---
print(results.short_summary())
print(results.get_estimated_parameters())

# --- print value of time ---
params = results.get_estimated_parameters()
vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_LOW']['Value']
print("\nValue of Time for HH <$100k: " + str(round(vot, 2)))

vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_HI']['Value']
print("Value of Time for HH $100k+: " + str(round(vot, 2)))

Results for model mnl_mode_choice
Nbr of parameters:		7
Sample size:			160922
Excluded data:			0
Null log likelihood:		-118890.3
Final log likelihood:		-63429.43
Likelihood ratio test (null):		110921.7
Rho square (null):			0.466
Rho bar square (null):			0.466
Akaike Information Criterion:	126872.9
Bayesian Information Criterion:	126942.8

                         Value  Rob. Std err  Rob. t-test  Rob. p-value
ASC_TRANSIT           1.936600      0.029147    66.441843           0.0
ASC_TRANSIT_DOWNTOWN -0.815453      0.033917   -24.042743           0.0
ASC_WALK              3.881068      0.032560   119.199111           0.0
ASC_WALK_DOWNTOWN    -1.275538      0.034290   -37.198485           0.0
B_COST_HI            -0.051764      0.003711   -13.948551           0.0
B_COST_LOW           -0.057176      0.002897   -19.736567           0.0
B_TIME               -0.024072      0.000674   -35.688880           0.0

Value of Time for HH <$100k: 25.26
Value of Time for HH $100k+: 27.9


In [36]:
# try separate walk and transit constants within downtown, but only based on origin

# include the transit fare
# weighted estimation with zonal incomes
# Add constant segmented by income
# use observed TNC time/fare where available

# --- coefficients ---
# Name, starting value, lower bound, upper bound, status (0=estimate, 1=fixed)
ASC_TRANSIT = Beta('ASC_TRANSIT', 0, None, None, 0)
ASC_WALK    = Beta('ASC_WALK',    0, None, None, 0)    
B_TIME      = Beta('B_TIME',      0, None, None, 0)   # generic, shared across modes
B_COST_LOW  = Beta('B_COST_LOW',  0, None, None, 0)   # cost coefficient for lower income travelers
B_COST_HI   = Beta('B_COST_HI',   0, None, None, 0)   # cost coefficient for higher income travelers

ASC_TRANSIT_DOWNTOWN = Beta('ASC_TRANSIT_DOWNTOWN', 0, None, None, 0) 
ASC_WALK_DOWNTOWN = Beta('ASC_WALK_DOWNTOWN', 0, None, None, 0) 

# --- variables ---
tnc_time     = Variable('tnc_time_3')
transit_time = Variable('transit_time')
walk_time    = Variable('walk_time')
tnc_fare     = Variable('tnc_fare_3')
transit_fare = Variable('transit_fare')

hh_share_inc_under_100k = Variable('hh_share_inc_under_100k')
hh_share_inc_over_100k = Variable('hh_share_inc_over_100k')

o_downtown = Variable('o_downtown')
d_downtown = Variable('d_downtown')

CHOICE       = Variable('CHOICE')

# --- utility equations ---
V_tnc     =               B_TIME * tnc_time     + B_COST_LOW * tnc_fare * hh_share_inc_under_100k + B_COST_HI * tnc_fare * hh_share_inc_over_100k         
V_transit = ASC_TRANSIT + B_TIME * transit_time + B_COST_LOW * transit_fare * hh_share_inc_under_100k + B_COST_HI * transit_fare * hh_share_inc_over_100k + ASC_TRANSIT_DOWNTOWN * o_downtown 
V_walk    = ASC_WALK    + B_TIME * walk_time + ASC_WALK_DOWNTOWN * o_downtown 

# specify which equations align with which alternatives, and the availability of each alternative
V  = {1: V_tnc, 2: V_transit, 3: V_walk}
avail = {1: 1, 2: Variable('transit_avail'), 3: Variable('walk_avail')}         

# --- estimate ---
logprob = models.loglogit(V, avail, CHOICE)
the_biogeme = bio.BIOGEME(db, {'loglike' : logprob, 'weight' : Variable('normalized_weights')})
the_biogeme.modelName = 'mnl_mode_choice'
the_biogeme.calculate_null_loglikelihood(avail=avail)
results = the_biogeme.estimate()

# --- print results ---
print(results.short_summary())
print(results.get_estimated_parameters())

# --- print value of time ---
params = results.get_estimated_parameters()
vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_LOW']['Value']
print("\nValue of Time for HH <$100k: " + str(round(vot, 2)))

vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_HI']['Value']
print("Value of Time for HH $100k+: " + str(round(vot, 2)))

Results for model mnl_mode_choice
Nbr of parameters:		7
Sample size:			160922
Excluded data:			0
Null log likelihood:		-118890.3
Final log likelihood:		-63621.45
Likelihood ratio test (null):		110537.7
Rho square (null):			0.465
Rho bar square (null):			0.465
Akaike Information Criterion:	127256.9
Bayesian Information Criterion:	127326.8

                         Value  Rob. Std err  Rob. t-test  Rob. p-value
ASC_TRANSIT           1.797343      0.027004    66.557283           0.0
ASC_TRANSIT_DOWNTOWN -0.357081      0.030785   -11.599293           0.0
ASC_WALK              3.755678      0.030170   124.485539           0.0
ASC_WALK_DOWNTOWN    -0.900033      0.034478   -26.104344           0.0
B_COST_HI            -0.066705      0.004131   -16.148203           0.0
B_COST_LOW           -0.061086      0.002974   -20.539919           0.0
B_TIME               -0.022651      0.000663   -34.161920           0.0

Value of Time for HH <$100k: 22.25
Value of Time for HH $100k+: 20.37


In [37]:
# try using 10 minute pickup time outside of downtown area
# we can clean this up in the estimation file, but for now just see what it does

# include the transit fare
# weighted estimation with zonal incomes
# Add constant segmented by income
# use observed TNC time/fare where available

# --- coefficients ---
# Name, starting value, lower bound, upper bound, status (0=estimate, 1=fixed)
ASC_TRANSIT = Beta('ASC_TRANSIT', 0, None, None, 0)
ASC_WALK    = Beta('ASC_WALK',    0, None, None, 0)    
B_TIME      = Beta('B_TIME',      0, None, None, 0)   # generic, shared across modes
B_COST_LOW  = Beta('B_COST_LOW',  0, None, None, 0)   # cost coefficient for lower income travelers
B_COST_HI   = Beta('B_COST_HI',   0, None, None, 0)   # cost coefficient for higher income travelers

# --- variables ---
tnc_time     = Variable('tnc_time_3')
transit_time = Variable('transit_time')
walk_time    = Variable('walk_time')
tnc_fare     = Variable('tnc_fare_3')
transit_fare = Variable('transit_fare')

hh_share_inc_under_100k = Variable('hh_share_inc_under_100k')
hh_share_inc_over_100k = Variable('hh_share_inc_over_100k')

o_downtown = Variable('o_downtown')

CHOICE       = Variable('CHOICE')

# --- utility equations ---
V_tnc     =               B_TIME * tnc_time     + B_COST_LOW * tnc_fare * hh_share_inc_under_100k + B_COST_HI * tnc_fare * hh_share_inc_over_100k          + B_TIME * 5 * (1-o_downtown)
V_transit = ASC_TRANSIT + B_TIME * transit_time + B_COST_LOW * transit_fare * hh_share_inc_under_100k + B_COST_HI * transit_fare * hh_share_inc_over_100k 
V_walk    = ASC_WALK    + B_TIME * walk_time    

# specify which equations align with which alternatives, and the availability of each alternative
V  = {1: V_tnc, 2: V_transit, 3: V_walk}
avail = {1: 1, 2: Variable('transit_avail'), 3: Variable('walk_avail')}         

# --- estimate ---
logprob = models.loglogit(V, avail, CHOICE)
the_biogeme = bio.BIOGEME(db, {'loglike' : logprob, 'weight' : Variable('normalized_weights')})
the_biogeme.modelName = 'mnl_mode_choice'
the_biogeme.calculate_null_loglikelihood(avail=avail)
results = the_biogeme.estimate()

# --- print results ---
print(results.short_summary())
print(results.get_estimated_parameters())

# --- print value of time ---
params = results.get_estimated_parameters()
vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_LOW']['Value']
print("\nValue of Time for HH <$100k: " + str(round(vot, 2)))

vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_HI']['Value']
print("Value of Time for HH $100k+: " + str(round(vot, 2)))

Results for model mnl_mode_choice
Nbr of parameters:		5
Sample size:			160922
Excluded data:			0
Null log likelihood:		-118890.3
Final log likelihood:		-64037.14
Likelihood ratio test (null):		109706.3
Rho square (null):			0.461
Rho bar square (null):			0.461
Akaike Information Criterion:	128084.3
Bayesian Information Criterion:	128134.2

                Value  Rob. Std err  Rob. t-test  Rob. p-value
ASC_TRANSIT  1.632913      0.024794    65.858818           0.0
ASC_WALK     3.434335      0.028869   118.963669           0.0
B_COST_HI   -0.045262      0.003428   -13.203951           0.0
B_COST_LOW  -0.074767      0.002972   -25.158532           0.0
B_TIME      -0.022518      0.000671   -33.541017           0.0

Value of Time for HH <$100k: 18.07
Value of Time for HH $100k+: 29.85


In [38]:
# what if we do a separate VOT for downtown trips

# include the transit fare
# weighted estimation with zonal incomes
# Add constant segmented by income
# use observed TNC time/fare where available

# --- coefficients ---
# Name, starting value, lower bound, upper bound, status (0=estimate, 1=fixed)
ASC_TRANSIT = Beta('ASC_TRANSIT', 0, None, None, 0)
ASC_WALK    = Beta('ASC_WALK',    0, None, None, 0)    
B_TIME      = Beta('B_TIME',      0, None, None, 0)   # generic, shared across modes
B_COST_LOW  = Beta('B_COST_LOW',  0, None, None, 0)   # cost coefficient for lower income travelers
B_COST_HI   = Beta('B_COST_HI',   0, None, None, 0)   # cost coefficient for higher income travelers
B_COST_DOWNTOWN = Beta('B_COST_DOWNTOWN', 0, None, None, 0) 

# --- variables ---
tnc_time     = Variable('tnc_time_3')
transit_time = Variable('transit_time')
walk_time    = Variable('walk_time')
tnc_fare     = Variable('tnc_fare_3')
transit_fare = Variable('transit_fare')

hh_share_inc_under_100k = Variable('hh_share_inc_under_100k')
hh_share_inc_over_100k = Variable('hh_share_inc_over_100k')

o_downtown = Variable('o_downtown')
d_downtown = Variable('d_downtown')

CHOICE       = Variable('CHOICE')

# --- utility equations ---
V_tnc     =               B_TIME * tnc_time     + B_COST_LOW * tnc_fare * hh_share_inc_under_100k + B_COST_HI * tnc_fare * hh_share_inc_over_100k + B_COST_DOWNTOWN * tnc_fare * o_downtown         
V_transit = ASC_TRANSIT + B_TIME * transit_time + B_COST_LOW * transit_fare * hh_share_inc_under_100k + B_COST_HI * transit_fare * hh_share_inc_over_100k + B_COST_DOWNTOWN * transit_fare * o_downtown
V_walk    = ASC_WALK    + B_TIME * walk_time    

# specify which equations align with which alternatives, and the availability of each alternative
V  = {1: V_tnc, 2: V_transit, 3: V_walk}
avail = {1: 1, 2: Variable('transit_avail'), 3: Variable('walk_avail')}         

# --- estimate ---
logprob = models.loglogit(V, avail, CHOICE)
the_biogeme = bio.BIOGEME(db, {'loglike' : logprob, 'weight' : Variable('normalized_weights')})
the_biogeme.modelName = 'mnl_mode_choice'
the_biogeme.calculate_null_loglikelihood(avail=avail)
results = the_biogeme.estimate()

# --- print results ---
print(results.short_summary())
print(results.get_estimated_parameters())

# --- print value of time ---
params = results.get_estimated_parameters()
vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_LOW']['Value']
print("\nValue of Time for HH <$100k: " + str(round(vot, 2)))

vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_HI']['Value']
print("Value of Time for HH $100k+: " + str(round(vot, 2)))


vot = 60 * params.loc['B_TIME']['Value'] / (params.loc['B_COST_LOW']['Value'] + params.loc['B_COST_DOWNTOWN']['Value'])
print("\nValue of Time for HH <$100k and downtown: " + str(round(vot, 2)))

vot = 60 * params.loc['B_TIME']['Value'] / (params.loc['B_COST_HI']['Value'] + params.loc['B_COST_DOWNTOWN']['Value'])
print("Value of Time for HH $100k+ and downtown: " + str(round(vot, 2)))


Results for model mnl_mode_choice
Nbr of parameters:		6
Sample size:			160922
Excluded data:			0
Null log likelihood:		-118890.3
Final log likelihood:		-63968.75
Likelihood ratio test (null):		109843
Rho square (null):			0.462
Rho bar square (null):			0.462
Akaike Information Criterion:	127949.5
Bayesian Information Criterion:	128009.4

                    Value  Rob. Std err  Rob. t-test  Rob. p-value
ASC_TRANSIT      1.695995      0.025517    66.464745           0.0
ASC_WALK         3.476186      0.029739   116.888577           0.0
B_COST_DOWNTOWN  0.038597      0.003020    12.782381           0.0
B_COST_HI       -0.084933      0.005694   -14.915257           0.0
B_COST_LOW      -0.069873      0.002997   -23.318167           0.0
B_TIME          -0.023434      0.000680   -34.452761           0.0

Value of Time for HH <$100k: 20.12
Value of Time for HH $100k+: 16.55

Value of Time for HH <$100k and downtown: 44.96
Value of Time for HH $100k+ and downtown: 30.34


In [39]:
# a single downtown VOT

# include the transit fare
# weighted estimation with zonal incomes
# Add constant segmented by income
# use observed TNC time/fare where available

# --- coefficients ---
# Name, starting value, lower bound, upper bound, status (0=estimate, 1=fixed)
ASC_TRANSIT = Beta('ASC_TRANSIT', 0, None, None, 0)
ASC_WALK    = Beta('ASC_WALK',    0, None, None, 0)    
B_TIME      = Beta('B_TIME',      0, None, None, 0)   # generic, shared across modes
B_COST_LOW  = Beta('B_COST_LOW',  0, None, None, 0)   # cost coefficient for lower income travelers
B_COST_HI   = Beta('B_COST_HI',   0, None, None, 0)   # cost coefficient for higher income travelers
B_COST_DOWNTOWN = Beta('B_COST_DOWNTOWN', 0, None, None, 0) 

# --- variables ---
tnc_time     = Variable('tnc_time_3')
transit_time = Variable('transit_time')
walk_time    = Variable('walk_time')
tnc_fare     = Variable('tnc_fare_3')
transit_fare = Variable('transit_fare')

hh_share_inc_under_100k = Variable('hh_share_inc_under_100k')
hh_share_inc_over_100k = Variable('hh_share_inc_over_100k')

o_downtown = Variable('o_downtown')
d_downtown = Variable('d_downtown')

CHOICE       = Variable('CHOICE')

# --- utility equations ---
V_tnc     =               B_TIME * tnc_time     + B_COST_LOW * tnc_fare * hh_share_inc_under_100k + B_COST_HI * tnc_fare * hh_share_inc_over_100k * (1-o_downtown) + B_COST_DOWNTOWN * tnc_fare * o_downtown         
V_transit = ASC_TRANSIT + B_TIME * transit_time + B_COST_LOW * transit_fare * hh_share_inc_under_100k + B_COST_HI * transit_fare * hh_share_inc_over_100k * (1-o_downtown) + B_COST_DOWNTOWN * transit_fare * o_downtown
V_walk    = ASC_WALK    + B_TIME * walk_time    

# specify which equations align with which alternatives, and the availability of each alternative
V  = {1: V_tnc, 2: V_transit, 3: V_walk}
avail = {1: 1, 2: Variable('transit_avail'), 3: Variable('walk_avail')}         

# --- estimate ---
logprob = models.loglogit(V, avail, CHOICE)
the_biogeme = bio.BIOGEME(db, {'loglike' : logprob, 'weight' : Variable('normalized_weights')})
the_biogeme.modelName = 'mnl_mode_choice'
the_biogeme.calculate_null_loglikelihood(avail=avail)
results = the_biogeme.estimate()

# --- print results ---
print(results.short_summary())
print(results.get_estimated_parameters())

# --- print value of time ---
params = results.get_estimated_parameters()
vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_LOW']['Value']
print("\nValue of Time for HH <$100k: " + str(round(vot, 2)))

vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_HI']['Value']
print("Value of Time for HH $100k+: " + str(round(vot, 2)))


vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_DOWNTOWN']['Value']
print("\nValue of Time for downtown origins: " + str(round(vot, 2)))



Results for model mnl_mode_choice
Nbr of parameters:		6
Sample size:			160922
Excluded data:			0
Null log likelihood:		-118890.3
Final log likelihood:		-63960.98
Likelihood ratio test (null):		109858.6
Rho square (null):			0.462
Rho bar square (null):			0.462
Akaike Information Criterion:	127934
Bayesian Information Criterion:	127993.9

                    Value  Rob. Std err  Rob. t-test  Rob. p-value
ASC_TRANSIT      1.685912      0.025893    65.111348           0.0
ASC_WALK         3.462060      0.030466   113.635779           0.0
B_COST_DOWNTOWN -0.017370      0.001999    -8.688773           0.0
B_COST_HI       -0.091545      0.006285   -14.566086           0.0
B_COST_LOW      -0.067652      0.002982   -22.689328           0.0
B_TIME          -0.023536      0.000681   -34.580697           0.0

Value of Time for HH <$100k: 20.87
Value of Time for HH $100k+: 15.43

Value of Time for downtown origins: 81.3


# Try different income allocation

What if we're smarter about how we allocate incomes to do it by time of day, or for non-downtown trips

In [44]:
# use the destination instead of the origin

# include the transit fare
# weighted estimation with zonal incomes
# Add constant segmented by income
# use observed TNC time/fare where available

# --- coefficients ---
# Name, starting value, lower bound, upper bound, status (0=estimate, 1=fixed)
ASC_TRANSIT = Beta('ASC_TRANSIT', 0, None, None, 0)
ASC_WALK    = Beta('ASC_WALK',    0, None, None, 0)    
B_TIME      = Beta('B_TIME',      0, None, None, 0)   # generic, shared across modes
B_COST_LOW  = Beta('B_COST_LOW',  0, None, None, 0)   # cost coefficient for lower income travelers
B_COST_HI   = Beta('B_COST_HI',   0, None, None, 0)   # cost coefficient for higher income travelers

# --- variables ---
tnc_time     = Variable('tnc_time_3')
transit_time = Variable('transit_time')
walk_time    = Variable('walk_time')
tnc_fare     = Variable('tnc_fare_3')
transit_fare = Variable('transit_fare')

hh_share_inc_under_100k_dropoff = Variable('hh_share_inc_under_100k_dropoff')
hh_share_inc_over_100k_dropoff = Variable('hh_share_inc_over_100k_dropoff')

CHOICE       = Variable('CHOICE')

# --- utility equations ---
V_tnc     =               B_TIME * tnc_time     + B_COST_LOW * tnc_fare * hh_share_inc_under_100k_dropoff + B_COST_HI * tnc_fare * hh_share_inc_over_100k_dropoff 
V_transit = ASC_TRANSIT + B_TIME * transit_time + B_COST_LOW * transit_fare * hh_share_inc_under_100k_dropoff + B_COST_HI * transit_fare * hh_share_inc_over_100k_dropoff 
V_walk    = ASC_WALK    + B_TIME * walk_time    

# specify which equations align with which alternatives, and the availability of each alternative
V  = {1: V_tnc, 2: V_transit, 3: V_walk}
avail = {1: 1, 2: Variable('transit_avail'), 3: Variable('walk_avail')}         

# --- estimate ---
logprob = models.loglogit(V, avail, CHOICE)
the_biogeme = bio.BIOGEME(db, {'loglike' : logprob, 'weight' : Variable('normalized_weights')})
the_biogeme.modelName = 'mnl_mode_choice'
the_biogeme.calculate_null_loglikelihood(avail=avail)
results = the_biogeme.estimate()

# --- print results ---
print(results.short_summary())
print(results.get_estimated_parameters())

# --- print value of time ---
params = results.get_estimated_parameters()
vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_LOW']['Value']
print("\nValue of Time for HH <$100k: " + str(round(vot, 2)))

vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_HI']['Value']
print("Value of Time for HH $100k+: " + str(round(vot, 2)))

Results for model mnl_mode_choice
Nbr of parameters:		5
Sample size:			160922
Excluded data:			0
Null log likelihood:		-118890.3
Final log likelihood:		-64110.79
Likelihood ratio test (null):		109559
Rho square (null):			0.461
Rho bar square (null):			0.461
Akaike Information Criterion:	128231.6
Bayesian Information Criterion:	128281.5

                Value  Rob. Std err  Rob. t-test  Rob. p-value
ASC_TRANSIT  1.697876      0.025066    67.737566           0.0
ASC_WALK     3.504628      0.028437   123.239765           0.0
B_COST_HI   -0.047972      0.003308   -14.503582           0.0
B_COST_LOW  -0.073452      0.003562   -20.623761           0.0
B_TIME      -0.021572      0.000650   -33.197953           0.0

Value of Time for HH <$100k: 17.62
Value of Time for HH $100k+: 26.98


In [45]:
# destination income allocation with the downtown constant back in.  

# include the transit fare
# weighted estimation with zonal incomes
# Add constant segmented by income
# use observed TNC time/fare where available

# --- coefficients ---
# Name, starting value, lower bound, upper bound, status (0=estimate, 1=fixed)
ASC_TRANSIT = Beta('ASC_TRANSIT', 0, None, None, 0)
ASC_WALK    = Beta('ASC_WALK',    0, None, None, 0)    
B_TIME      = Beta('B_TIME',      0, None, None, 0)   # generic, shared across modes
B_COST_LOW  = Beta('B_COST_LOW',  0, None, None, 0)   # cost coefficient for lower income travelers
B_COST_HI   = Beta('B_COST_HI',   0, None, None, 0)   # cost coefficient for higher income travelers

ASC_TNC_DOWNTOWN = Beta('ASC_TNC_DOWNTOWN', 0, None, None, 0) 

# --- variables ---
tnc_time     = Variable('tnc_time_3')
transit_time = Variable('transit_time')
walk_time    = Variable('walk_time')
tnc_fare     = Variable('tnc_fare_3')
transit_fare = Variable('transit_fare')

hh_share_inc_under_100k_dropoff = Variable('hh_share_inc_under_100k_dropoff')
hh_share_inc_over_100k_dropoff = Variable('hh_share_inc_over_100k_dropoff')

o_downtown = Variable('o_downtown')
d_downtown = Variable('d_downtown')

CHOICE       = Variable('CHOICE')

# --- utility equations ---
V_tnc     =               B_TIME * tnc_time     + B_COST_LOW * tnc_fare * hh_share_inc_under_100k_dropoff + B_COST_HI * tnc_fare * hh_share_inc_over_100k_dropoff  + ASC_TNC_DOWNTOWN * o_downtown * d_downtown
V_transit = ASC_TRANSIT + B_TIME * transit_time + B_COST_LOW * transit_fare * hh_share_inc_under_100k_dropoff + B_COST_HI * transit_fare * hh_share_inc_over_100k_dropoff 
V_walk    = ASC_WALK    + B_TIME * walk_time    

# specify which equations align with which alternatives, and the availability of each alternative
V  = {1: V_tnc, 2: V_transit, 3: V_walk}
avail = {1: 1, 2: Variable('transit_avail'), 3: Variable('walk_avail')}         

# --- estimate ---
logprob = models.loglogit(V, avail, CHOICE)
the_biogeme = bio.BIOGEME(db, {'loglike' : logprob, 'weight' : Variable('normalized_weights')})
the_biogeme.modelName = 'mnl_mode_choice'
the_biogeme.calculate_null_loglikelihood(avail=avail)
results = the_biogeme.estimate()

# --- print results ---
print(results.short_summary())
print(results.get_estimated_parameters())

# --- print value of time ---
params = results.get_estimated_parameters()
vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_LOW']['Value']
print("\nValue of Time for HH <$100k: " + str(round(vot, 2)))

vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_HI']['Value']
print("Value of Time for HH $100k+: " + str(round(vot, 2)))

Results for model mnl_mode_choice
Nbr of parameters:		6
Sample size:			160922
Excluded data:			0
Null log likelihood:		-118890.3
Final log likelihood:		-63653.69
Likelihood ratio test (null):		110473.2
Rho square (null):			0.465
Rho bar square (null):			0.465
Akaike Information Criterion:	127319.4
Bayesian Information Criterion:	127379.3

                     Value  Rob. Std err  Rob. t-test  Rob. p-value
ASC_TNC_DOWNTOWN  1.030098      0.032331    31.861374           0.0
ASC_TRANSIT       1.981939      0.028409    69.764838           0.0
ASC_WALK          3.798915      0.031500   120.599585           0.0
B_COST_HI        -0.053833      0.003520   -15.292247           0.0
B_COST_LOW       -0.054180      0.003423   -15.830431           0.0
B_TIME           -0.024778      0.000677   -36.574294           0.0

Value of Time for HH <$100k: 27.44
Value of Time for HH $100k+: 27.62


Little VOT difference here still.  Downtown is weird. 

In [46]:
# use the destination income allocation only if the origin is downtown

# include the transit fare
# weighted estimation with zonal incomes
# Add constant segmented by income
# use observed TNC time/fare where available

# --- coefficients ---
# Name, starting value, lower bound, upper bound, status (0=estimate, 1=fixed)
ASC_TRANSIT = Beta('ASC_TRANSIT', 0, None, None, 0)
ASC_WALK    = Beta('ASC_WALK',    0, None, None, 0)    
B_TIME      = Beta('B_TIME',      0, None, None, 0)   # generic, shared across modes
B_COST_LOW  = Beta('B_COST_LOW',  0, None, None, 0)   # cost coefficient for lower income travelers
B_COST_HI   = Beta('B_COST_HI',   0, None, None, 0)   # cost coefficient for higher income travelers

# --- variables ---
tnc_time     = Variable('tnc_time_3')
transit_time = Variable('transit_time')
walk_time    = Variable('walk_time')
tnc_fare     = Variable('tnc_fare_3')
transit_fare = Variable('transit_fare')

hh_share_inc_under_100k = Variable('hh_share_inc_under_100k')
hh_share_inc_over_100k = Variable('hh_share_inc_over_100k')
hh_share_inc_under_100k_dropoff = Variable('hh_share_inc_under_100k_dropoff')
hh_share_inc_over_100k_dropoff = Variable('hh_share_inc_over_100k_dropoff')

o_downtown = Variable('o_downtown')
d_downtown = Variable('d_downtown')

CHOICE       = Variable('CHOICE')

# --- utility equations ---
V_tnc     = (              B_TIME * tnc_time     
                        + B_COST_LOW * tnc_fare * hh_share_inc_under_100k * (1-o_downtown) + B_COST_HI * tnc_fare * hh_share_inc_over_100k * (1-o_downtown) 
                        + B_COST_LOW * tnc_fare * hh_share_inc_under_100k_dropoff * (o_downtown) + B_COST_HI * tnc_fare * hh_share_inc_over_100k_dropoff * (o_downtown) 
                        )
V_transit = (ASC_TRANSIT + B_TIME * transit_time 
                        + B_COST_LOW * transit_fare * hh_share_inc_under_100k * (1-o_downtown) + B_COST_HI * transit_fare * hh_share_inc_over_100k * (1-o_downtown) 
                        + B_COST_LOW * transit_fare * hh_share_inc_under_100k_dropoff * (o_downtown) + B_COST_HI * transit_fare * hh_share_inc_over_100k_dropoff * (o_downtown) 
                        )
V_walk    = ASC_WALK    + B_TIME * walk_time    

# specify which equations align with which alternatives, and the availability of each alternative
V  = {1: V_tnc, 2: V_transit, 3: V_walk}
avail = {1: 1, 2: Variable('transit_avail'), 3: Variable('walk_avail')}         

# --- estimate ---
logprob = models.loglogit(V, avail, CHOICE)
the_biogeme = bio.BIOGEME(db, {'loglike' : logprob, 'weight' : Variable('normalized_weights')})
the_biogeme.modelName = 'mnl_mode_choice'
the_biogeme.calculate_null_loglikelihood(avail=avail)
results = the_biogeme.estimate()

# --- print results ---
print(results.short_summary())
print(results.get_estimated_parameters())

# --- print value of time ---
params = results.get_estimated_parameters()
vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_LOW']['Value']
print("\nValue of Time for HH <$100k: " + str(round(vot, 2)))

vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_HI']['Value']
print("Value of Time for HH $100k+: " + str(round(vot, 2)))

Results for model mnl_mode_choice
Nbr of parameters:		5
Sample size:			160922
Excluded data:			0
Null log likelihood:		-118890.3
Final log likelihood:		-64062.58
Likelihood ratio test (null):		109655.4
Rho square (null):			0.461
Rho bar square (null):			0.461
Akaike Information Criterion:	128135.2
Bayesian Information Criterion:	128185.1

                Value  Rob. Std err  Rob. t-test  Rob. p-value
ASC_TRANSIT  1.727412      0.025456    67.858305  0.000000e+00
ASC_WALK     3.555816      0.029003   122.601965  0.000000e+00
B_COST_HI   -0.026347      0.004102    -6.423619  1.330722e-10
B_COST_LOW  -0.079804      0.002858   -27.920605  0.000000e+00
B_TIME      -0.021032      0.000660   -31.865863  0.000000e+00

Value of Time for HH <$100k: 15.81
Value of Time for HH $100k+: 47.9


In [47]:
# use the destination income allocation only if the origin is downtown
# and add the downtown constant back in

# include the transit fare
# weighted estimation with zonal incomes
# Add constant segmented by income
# use observed TNC time/fare where available

# --- coefficients ---
# Name, starting value, lower bound, upper bound, status (0=estimate, 1=fixed)
ASC_TRANSIT = Beta('ASC_TRANSIT', 0, None, None, 0)
ASC_WALK    = Beta('ASC_WALK',    0, None, None, 0)    
B_TIME      = Beta('B_TIME',      0, None, None, 0)   # generic, shared across modes
B_COST_LOW  = Beta('B_COST_LOW',  0, None, None, 0)   # cost coefficient for lower income travelers
B_COST_HI   = Beta('B_COST_HI',   0, None, None, 0)   # cost coefficient for higher income travelers

ASC_TNC_DOWNTOWN = Beta('ASC_TNC_DOWNTOWN', 0, None, None, 0) 

# --- variables ---
tnc_time     = Variable('tnc_time_3')
transit_time = Variable('transit_time')
walk_time    = Variable('walk_time')
tnc_fare     = Variable('tnc_fare_3')
transit_fare = Variable('transit_fare')

hh_share_inc_under_100k = Variable('hh_share_inc_under_100k')
hh_share_inc_over_100k = Variable('hh_share_inc_over_100k')
hh_share_inc_under_100k_dropoff = Variable('hh_share_inc_under_100k_dropoff')
hh_share_inc_over_100k_dropoff = Variable('hh_share_inc_over_100k_dropoff')

o_downtown = Variable('o_downtown')
d_downtown = Variable('d_downtown')

CHOICE       = Variable('CHOICE')

# --- utility equations ---
V_tnc     = (              B_TIME * tnc_time     
                        + B_COST_LOW * tnc_fare * hh_share_inc_under_100k * (1-o_downtown) + B_COST_HI * tnc_fare * hh_share_inc_over_100k * (1-o_downtown) 
                        + B_COST_LOW * tnc_fare * hh_share_inc_under_100k_dropoff * (o_downtown) + B_COST_HI * tnc_fare * hh_share_inc_over_100k_dropoff * (o_downtown) 
                        + ASC_TNC_DOWNTOWN * o_downtown * d_downtown
                        )
V_transit = (ASC_TRANSIT + B_TIME * transit_time 
                        + B_COST_LOW * transit_fare * hh_share_inc_under_100k * (1-o_downtown) + B_COST_HI * transit_fare * hh_share_inc_over_100k * (1-o_downtown) 
                        + B_COST_LOW * transit_fare * hh_share_inc_under_100k_dropoff * (o_downtown) + B_COST_HI * transit_fare * hh_share_inc_over_100k_dropoff * (o_downtown) 
                        )
V_walk    = ASC_WALK    + B_TIME * walk_time    

# specify which equations align with which alternatives, and the availability of each alternative
V  = {1: V_tnc, 2: V_transit, 3: V_walk}
avail = {1: 1, 2: Variable('transit_avail'), 3: Variable('walk_avail')}         

# --- estimate ---
logprob = models.loglogit(V, avail, CHOICE)
the_biogeme = bio.BIOGEME(db, {'loglike' : logprob, 'weight' : Variable('normalized_weights')})
the_biogeme.modelName = 'mnl_mode_choice'
the_biogeme.calculate_null_loglikelihood(avail=avail)
results = the_biogeme.estimate()

# --- print results ---
print(results.short_summary())
print(results.get_estimated_parameters())

# --- print value of time ---
params = results.get_estimated_parameters()
vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_LOW']['Value']
print("\nValue of Time for HH <$100k: " + str(round(vot, 2)))

vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_HI']['Value']
print("Value of Time for HH $100k+: " + str(round(vot, 2)))

Results for model mnl_mode_choice
Nbr of parameters:		6
Sample size:			160922
Excluded data:			0
Null log likelihood:		-118890.3
Final log likelihood:		-63646.1
Likelihood ratio test (null):		110488.3
Rho square (null):			0.465
Rho bar square (null):			0.465
Akaike Information Criterion:	127304.2
Bayesian Information Criterion:	127364.1

                     Value  Rob. Std err  Rob. t-test  Rob. p-value
ASC_TNC_DOWNTOWN  0.993548      0.032919    30.181315           0.0
ASC_TRANSIT       1.988237      0.028605    69.506329           0.0
ASC_WALK          3.812920      0.032055   118.949993           0.0
B_COST_HI        -0.040601      0.004467    -9.089915           0.0
B_COST_LOW       -0.061161      0.002795   -21.882549           0.0
B_TIME           -0.024635      0.000681   -36.177131           0.0

Value of Time for HH <$100k: 24.17
Value of Time for HH $100k+: 36.41


# Hold other coefficients constant and estimate just the downtown constant

In [49]:
# start from initial model.

# include a constant for trips within downtown

# include the transit fare
# weighted estimation with zonal incomes
# Add constant segmented by income
# use observed TNC time/fare where available

# --- coefficients ---
# Name, starting value, lower bound, upper bound, status (0=estimate, 1=fixed)
ASC_TRANSIT = Beta('ASC_TRANSIT', 1.706168, None, None, 0)
ASC_WALK    = Beta('ASC_WALK',    3.520829, None, None, 0)    
B_TIME      = Beta('B_TIME',      -0.021315, -0.021315, -0.021315, 0)   # generic, shared across modes
B_COST_LOW  = Beta('B_COST_LOW',  -0.077116, -0.077116, -0.077116, 0)   # cost coefficient for lower income travelers
B_COST_HI   = Beta('B_COST_HI',   -0.039206, -0.039206, -0.039206, 0)   # cost coefficient for higher income travelers

ASC_TNC_DOWNTOWN = Beta('ASC_TNC_DOWNTOWN', 0, None, None, 0) 

# --- variables ---
tnc_time     = Variable('tnc_time_3')
transit_time = Variable('transit_time')
walk_time    = Variable('walk_time')
tnc_fare     = Variable('tnc_fare_3')
transit_fare = Variable('transit_fare')

hh_share_inc_under_100k = Variable('hh_share_inc_under_100k')
hh_share_inc_over_100k = Variable('hh_share_inc_over_100k')

o_downtown = Variable('o_downtown')
d_downtown = Variable('d_downtown')

CHOICE       = Variable('CHOICE')

# --- utility equations ---
V_tnc     = (             B_TIME * tnc_time     
                        + B_COST_LOW * tnc_fare * hh_share_inc_under_100k + B_COST_HI * tnc_fare * hh_share_inc_over_100k         
                         + ASC_TNC_DOWNTOWN * o_downtown * d_downtown )
V_transit = ASC_TRANSIT + B_TIME * transit_time + B_COST_LOW * transit_fare * hh_share_inc_under_100k + B_COST_HI * transit_fare * hh_share_inc_over_100k 
V_walk    = ASC_WALK    + B_TIME * walk_time    

# specify which equations align with which alternatives, and the availability of each alternative
V  = {1: V_tnc, 2: V_transit, 3: V_walk}
avail = {1: 1, 2: Variable('transit_avail'), 3: Variable('walk_avail')}         

# --- estimate ---
logprob = models.loglogit(V, avail, CHOICE)
the_biogeme = bio.BIOGEME(db, {'loglike' : logprob, 'weight' : Variable('normalized_weights')})
the_biogeme.modelName = 'mnl_mode_choice'
the_biogeme.calculate_null_loglikelihood(avail=avail)
results = the_biogeme.estimate()

# --- print results ---
print(results.short_summary())
print(results.get_estimated_parameters())

# --- print value of time ---
params = results.get_estimated_parameters()
vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_LOW']['Value']
print("\nValue of Time for HH <$100k: " + str(round(vot, 2)))

vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_HI']['Value']
print("Value of Time for HH $100k+: " + str(round(vot, 2)))

Results for model mnl_mode_choice
Nbr of parameters:		6
Sample size:			160922
Excluded data:			0
Null log likelihood:		-118890.3
Final log likelihood:		-63693.56
Likelihood ratio test (null):		110393.4
Rho square (null):			0.464
Rho bar square (null):			0.464
Akaike Information Criterion:	127399.1
Bayesian Information Criterion:	127459.1

                     Value  Active bound  Rob. Std err  Rob. t-test  \
ASC_TNC_DOWNTOWN  0.900369           0.0      0.032036    28.105292   
ASC_TRANSIT       1.824339           0.0      0.027698    65.866023   
ASC_WALK          3.662597           0.0      0.030753   119.096710   
B_COST_HI        -0.039206           1.0      0.003535   -11.091575   
B_COST_LOW       -0.077116           1.0      0.003234   -23.845248   
B_TIME           -0.021315           1.0      0.000662   -32.205702   

                  Rob. p-value  
ASC_TNC_DOWNTOWN           0.0  
ASC_TRANSIT                0.0  
ASC_WALK                   0.0  
B_COST_HI                  0.

The last one is a decent option as well.  The same as our initial preferred model, but with calibrated downtown constant.  